In [1]:
import os
import glob
import h5py
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda
from tqdm import tqdm
from sklearn.neighbors import BallTree

In [2]:
df = pd.read_pickle('./df_comp_kmeans_mod.pkl')
df

,sys_name,frame,n_res,dist,n_contacts,rmsd,rmsd_nsp10,rmsd_nsp16,labels,mod
0,comp_0,0,415,32.605477,143,0.000503,0.000501,0.000501,453,1
1,comp_0,1,415,32.323373,144,1.128383,1.001219,0.960327,228,0
2,comp_0,2,415,31.932924,156,1.166511,1.258622,0.889202,448,0
3,comp_0,3,415,32.521470,149,1.076905,0.862441,1.087489,59,0
4,comp_0,4,415,32.193099,157,1.476163,1.624638,1.130987,183,0
...,...,...,...,...,...,...,...,...,...,...
83995,comp_100,3995,415,39.647390,107,10.251899,4.789822,2.935981,283,4
83996,comp_100,3996,415,39.943104,114,10.175004,4.315601,3.046836,102,4
83997,comp_100,3997,415,39.952250,113,10.413391,4.641341,2.946265,72,4
83998,comp_100,3998,415,40.436388,108,10.257934,4.416924,3.048530,72,4


In [5]:
h5_cm = h5py.File('../cvae_comp/cvae_40/latent.h5', 'r')

cm_predict = h5_cm['latent']

In [6]:
pdb_save = 'pdb_save_clustering_comp'
os.makedirs(pdb_save, exist_ok=True)

In [7]:
pdb_store_path = '../../traj_save'

In [16]:
def get_pdb_name(sys_name): 
    name_split = sys_name.split('_')
    if len(name_split) == 3: 
        lig_name = name_split[1]
        pdb = f'{pdb_store_path}/nsp10_16_dist_{lig_name}/{sys_name}.pdb'
        if not os.path.exists(pdb): 
            print(pdb)
    elif len(name_split[-1]) == 3: 
        pdb = f'{pdb_store_path}/nsp10_16_dist/{sys_name}.pdb'
        if not os.path.exists(pdb): 
            print(pdb)
    else: 
        pdb = f'{pdb_store_path}/Nsp10_Nsp16/{sys_name}.pdb'
        if not os.path.exists(pdb): 
            print(pdb)
    return pdb

In [17]:
def write_pdb_frame(sys_name, frame, save_pdb_path): 
    pdb = get_pdb_name(sys_name)
    dcd = pdb.replace('pdb', 'dcd')
    
    mda_u = mda.Universe(pdb, dcd)
    mda_u.trajectory[frame]
    mda_u.atoms.write(save_pdb_path)

In [26]:
n_pdbs = 10

for mod in tqdm(sorted(df['mod'].unique())): 
    sub_df = df[df['mod'] == mod]
#     break
#     mod_coords = sub_df[[col for col in sub_df.columns if col.startswith('lat')]]
    mod_coords = cm_predict[sub_df.index.to_numpy()]
    mod_center = np.mean(mod_coords, axis=0)
    
    tree = BallTree(mod_coords, leaf_size=40) 
    dist, inds = tree.query([mod_center], k=n_pdbs)
    
    mod_save_path = f'{pdb_save}/cluster_{mod:03}'
    os.makedirs(mod_save_path, exist_ok=True)
    
    inds = inds.ravel() 
    for i, ind in enumerate(inds):
        row = sub_df.iloc[ind]
        sys_name = row['sys_name']
        frame = int(row['frame'])
        t_ind = int(row.name)
        save_pdb_path = f'{mod_save_path}/{i:03}_{t_ind}_{sys_name}_{frame:06}.pdb'
        write_pdb_frame(sys_name, frame, save_pdb_path)
        

  0%|                                                                                                                                                                                                                | 0/35 [00:00<?, ?it/s]/homes/heng.ma/miniconda3/envs/MD_ff/lib/python3.8/site-packages/MDAnalysis/coordinates/PDB.py:1126: UserWarning: Found missing chainIDs. Corresponding atoms will use value of 'X'
  warnings.warn("Found missing chainIDs."
  3%|█████▋                                                                                                                                                                                                  | 1/35 [00:05<02:51,  5.05s/it]/homes/heng.ma/miniconda3/envs/MD_ff/lib/python3.8/site-packages/MDAnalysis/coordinates/PDB.py:1126: UserWarning: Found missing chainIDs. Corresponding atoms will use value of 'X'
  warnings.warn("Found missing chainIDs."
  6%|███████████▍                                                                   

 51%|██████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                                                                | 18/35 [00:31<00:22,  1.33s/it]/homes/heng.ma/miniconda3/envs/MD_ff/lib/python3.8/site-packages/MDAnalysis/coordinates/PDB.py:1126: UserWarning: Found missing chainIDs. Corresponding atoms will use value of 'X'
  warnings.warn("Found missing chainIDs."
 54%|████████████████████████████████████████████████████████████████████████████████████████████████████████████                                                                                           | 19/35 [00:33<00:21,  1.32s/it]/homes/heng.ma/miniconda3/envs/MD_ff/lib/python3.8/site-packages/MDAnalysis/coordinates/PDB.py:1126: UserWarning: Found missing chainIDs. Corresponding atoms will use value of 'X'
  warnings.warn("Found missing chainIDs."
 57%|███████████████████████████████████████████████████████████████████████████████

In [15]:
sub_df

,sys_name,time_frame,rmsd_all,rmsd_nsp10,rmsd_nsp16,time,dist,n_contacts,labels,mod
0,comp,0,0.00050335,0.000508361,0.000501456,0,32.605477,143,212,0
1,comp,1,1.12838,1.43292,0.985207,0.05,32.323373,144,7,0
2,comp,2,1.16651,1.6247,0.929828,0.1,31.932924,156,118,0
3,comp,3,1.0769,1.01656,1.09943,0.15,32.521470,149,23,0
4,comp,4,1.47616,2.03988,1.18747,0.2,32.193099,157,329,0
...,...,...,...,...,...,...,...,...,...,...
343985,comp_sfg_080,3985,5.07839,8.01727,3.29526,199.25,32.316166,144,370,0
343989,comp_sfg_080,3989,5.14863,8.18076,3.29065,199.45,32.161904,148,237,0
343995,comp_sfg_080,3995,5.12106,8.09204,3.31595,199.75,32.317198,150,370,0
343998,comp_sfg_080,3998,5.35664,8.43882,3.49249,199.9,32.383680,141,370,0


In [17]:
sub_df.index.to_numpy()

array([     0,      1,      2, ..., 343995, 343998, 343999])

In [6]:
traj_dir = '../../traj_save/'
def get_pdb_path(sys_name): 
    lig_name = sys_name.split('_')[1] if len(sys_name.split('_')) == 3 else None
    lig_dir = 'nsp10_16_dist' + f'_{lig_name}' if lig_name else 'nsp10_16_dist'
    pdb_file = f'{traj_dir}/{lig_dir}/{sys_name}.pdb'
    if os.path.exists(pdb_file): 
        return pdb_file
    else: 
        raise("Mising file")
    

In [7]:
def write_pdb_file(sys_name, frame:int): 
    pdb = get_pdb_path(sys_name) 
    dcd = pdb.replace('.pdb', '.dcd')
    
    mda_u = mda.Universe(pdb, dcd)
    mda_u.trajectory[frame] 
    
    pdb_name = f'{sys_name}_{frame:05}.pdb'
    pdb_name = pdb_save + f'/{pdb_name}'
    mda_u.atoms.write(pdb_name)

In [8]:
output_pdbs = [['comp_sfg_095', 3500], ['comp_sam_095', 533], ['comp_sah_065', 3699],]

In [9]:
for i in output_pdbs: 
#     print(i)
    write_pdb_file(*i)